# LoRA — Low-Rank Adaptation of Large Language Models (2021)

_Papers · Transformers & LLMs_

**Fine-tune a frozen model by learning a tiny low-rank weight update instead of all the weights.**

---

This notebook is a practice scaffold for the **LoRA — Low-Rank Adaptation of Large Language Models (2021)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, copy
torch.manual_seed(0); np.random.seed(0)

# --- 0. Worked example: rank-1 LoRA update on a 4x3 layer; count trainable params. ---
B = torch.tensor([[1.],[0.],[2.],[-1.]])      # d x r  = 4 x 1
A = torch.tensor([[1., 0., -1.]])             # r x k  = 1 x 3
BA = B @ A                                    # 4 x 3, rank 1
print("worked: BA =\n", BA)                   # rows are multiples of [1,0,-1]
print("worked: (alpha/r)*BA row0  (alpha=2,r=1) =", (2.0/1.0)*BA[0])   # [2,0,-2]
print("worked: LoRA trainable params (r=1) = B(4*1)+A(1*3) =", 4*1+1*3,
      " vs full W = 4*3 =", 4*3)
# worked: (alpha/r)*BA row0 (alpha=2,r=1) = tensor([ 2.,  0., -2.])
# worked: LoRA trainable params (r=1) = 7  vs full W = 12


# --- 1. A small net, composed with torch.nn. Stands in for a "pretrained model". ---
D_IN, H, N_CLS = 8, 32, 4
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(D_IN, H)
        self.fc2  = nn.Linear(H, H)            # the layer we will adapt with LoRA (d=k=H)
        self.head = nn.Linear(H, N_CLS)
    def forward(self, x, lora=None):
        x = torch.relu(self.fc1(x))
        h = self.fc2(x)                        # W0 x  (frozen base)
        if lora is not None: h = h + lora(x)   # + (alpha/r) B A x
        return self.head(torch.relu(h))

# --- 2. Two tasks: pretrain on task A, then adapt to a DIFFERENT new task B. ---
def make_blobs(centers, n, noise=0.6, seed=0):
    rng = np.random.RandomState(seed); X, y = [], []
    for c, mu in enumerate(centers):
        X.append(rng.randn(n, D_IN)*noise + mu); y += [c]*n
    X = np.concatenate(X); y = np.array(y); p = rng.permutation(len(y))
    return X[p], y[p]
rc = np.random.RandomState(1)
centersA = rc.randn(N_CLS, D_IN)*2.0; centersB = rc.randn(N_CLS, D_IN)*2.0
XA, yA       = make_blobs(centersA, 400, seed=10)
XBtr, yBtr   = make_blobs(centersB,  60, seed=20)   # SMALL new-task train set
XBte, yBte   = make_blobs(centersB, 400, seed=21)
T  = lambda a: torch.tensor(a, dtype=torch.float32)
Ti = lambda a: torch.tensor(a, dtype=torch.long)
def acc(m, X, y, lora=None):
    with torch.no_grad():
        return (m(T(X), lora).argmax(1) == Ti(y)).float().mean().item()

net = Net(); opt = torch.optim.Adam(net.parameters(), lr=1e-2)
for _ in range(300):
    opt.zero_grad(); F.cross_entropy(net(T(XA)), Ti(yA)).backward(); opt.step()
print("\nfrozen net on base task A acc:", round(acc(net, XA, yA), 3))
print("frozen net (no adapt) on NEW task B acc:", round(acc(net, XBte, yBte), 3))

# --- 3. The NOVEL part, by hand: a LoRA module. B=0, A=Gaussian, scaled by alpha/r. ---
class LoRA(nn.Module):
    def __init__(self, d, k, r, alpha):
        super().__init__()
        self.A = nn.Parameter(torch.randn(r, k) * 0.01)   # random Gaussian
        self.B = nn.Parameter(torch.zeros(d, r))          # ZERO -> update is 0 at init
        self.scale = alpha / r                            # the alpha/r scaling (Sec 4.1)
    def forward(self, x):
        return (x @ self.A.t() @ self.B.t()) * self.scale # (alpha/r) * (B A) x

count = lambda ps: sum(p.numel() for p in ps if p.requires_grad)

# --- 4. Baseline: FULL fine-tuning (unfreeze everything) on the new task. ---
full = copy.deepcopy(net)
for p in full.parameters(): p.requires_grad_(True)
fopt = torch.optim.Adam(full.parameters(), lr=1e-2)
for _ in range(400):
    fopt.zero_grad(); F.cross_entropy(full(T(XBtr)), Ti(yBtr)).backward(); fopt.step()
full_n, full_a = count(list(full.parameters())), acc(full, XBte, yBte)
print("\nFULL fine-tune : trainable params = %4d  test acc = %.3f" % (full_n, full_a))

# --- 5. LoRA: freeze the net, train only B,A on the new task. Sweep rank r. ---
for r in [1, 2, 4, 8]:
    for p in net.parameters(): p.requires_grad_(False)   # W0 stays frozen
    lora = LoRA(H, H, r, alpha=8)
    lopt = torch.optim.Adam(lora.parameters(), lr=1e-2)  # ONLY LoRA params
    for _ in range(400):
        lopt.zero_grad(); F.cross_entropy(net(T(XBtr), lora), Ti(yBtr)).backward(); lopt.step()
    print("LoRA r=%d       : trainable params = %4d  test acc = %.3f"
          % (r, count(list(lora.parameters())), acc(net, XBte, yBte, lora)))

# --- 6. Ablation / check: at init (B=0) the update is exactly zero. ---
lora0 = LoRA(H, H, 4, 8)
print("\nLoRA update at init (B=0), max abs over a batch:",
      round(lora0(torch.randn(16, H)).abs().max().item(), 8), "(should be 0.0)")

# Our small run, not the paper's numbers. Typical output:
#   frozen net (no adapt) on NEW task B acc: 0.188
#   FULL fine-tune : trainable params = 1476  test acc = 1.000
#   LoRA r=1       : trainable params =   64  test acc = 0.640
#   LoRA r=2       : trainable params =  128  test acc = 0.992
#   LoRA r=4       : trainable params =  256  test acc = 0.999
#   LoRA r=8       : trainable params =  512  test acc = 0.998
#   LoRA update at init (B=0): 0.0

## Visualize the data & results

_Adapting a FROZEN net to a new task: how does held-out accuracy trade off against the number of trainable parameters, for full fine-tuning versus LoRA at ranks r = 1, 2, 4, 8?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, copy
torch.manual_seed(0); np.random.seed(0)

D_IN, H, N_CLS = 8, 32, 4
class Net(nn.Module):
    def __init__(s):
        super().__init__()
        s.fc1=nn.Linear(D_IN,H); s.fc2=nn.Linear(H,H); s.head=nn.Linear(H,N_CLS)
    def forward(s,x,lora=None):
        x=torch.relu(s.fc1(x)); h=s.fc2(x)
        if lora is not None: h=h+lora(x)
        return s.head(torch.relu(h))
class LoRA(nn.Module):
    def __init__(s,d,k,r,alpha):
        super().__init__()
        s.A=nn.Parameter(torch.randn(r,k)*0.01); s.B=nn.Parameter(torch.zeros(d,r)); s.scale=alpha/r
    def forward(s,x): return (x@s.A.t()@s.B.t())*s.scale

def blobs(centers,n,noise=0.6,seed=0):
    rng=np.random.RandomState(seed); X,y=[],[]
    for c,mu in enumerate(centers): X.append(rng.randn(n,D_IN)*noise+mu); y+=[c]*n
    X=np.concatenate(X); y=np.array(y); p=rng.permutation(len(y)); return X[p],y[p]
rc=np.random.RandomState(1); cA=rc.randn(N_CLS,D_IN)*2.0; cB=rc.randn(N_CLS,D_IN)*2.0
XA,yA=blobs(cA,400,seed=10); Xtr,ytr=blobs(cB,60,seed=20); Xte,yte=blobs(cB,400,seed=21)
T=lambda a: torch.tensor(a,dtype=torch.float32); Ti=lambda a: torch.tensor(a,dtype=torch.long)
def acc(m,X,y,lora=None):
    with torch.no_grad(): return (m(T(X),lora).argmax(1)==Ti(y)).float().mean().item()
cnt=lambda ps: sum(p.numel() for p in ps if p.requires_grad)

# pretrain base task, then freeze
net=Net(); o=torch.optim.Adam(net.parameters(),lr=1e-2)
for _ in range(300): o.zero_grad(); F.cross_entropy(net(T(XA)),Ti(yA)).backward(); o.step()
print("no-adapt new-task acc:", round(acc(net,Xte,yte),3))

# full fine-tuning baseline
full=copy.deepcopy(net)
for p in full.parameters(): p.requires_grad_(True)
fo=torch.optim.Adam(full.parameters(),lr=1e-2)
for _ in range(400): fo.zero_grad(); F.cross_entropy(full(T(Xtr)),Ti(ytr)).backward(); fo.step()
print("FULL:", cnt(list(full.parameters())), round(acc(full,Xte,yte),3))

# LoRA rank sweep
for r in [1,2,4,8]:
    for p in net.parameters(): p.requires_grad_(False)
    lora=LoRA(H,H,r,alpha=8); lo=torch.optim.Adam(lora.parameters(),lr=1e-2)
    for _ in range(400): lo.zero_grad(); F.cross_entropy(net(T(Xtr),lora),Ti(ytr)).backward(); lo.step()
    print("LoRA r=%d:"%r, cnt(list(lora.parameters())), round(acc(net,Xte,yte,lora),3))
# no-adapt new-task acc: 0.188
# FULL: 1476 1.0
# LoRA r=1: 64 0.64
# LoRA r=2: 128 0.992
# LoRA r=4: 256 0.999
# LoRA r=8: 512 0.998
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The rank sweep (ablation). You adapt a frozen net to a new task with LoRA at ranks
            $r \in \{1, 2, 4, 8\}$ and also with full fine-tuning. As $r$ grows, what happens to the number of
            trainable parameters and to accuracy? At what point does LoRA match full fine-tuning?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count trainable params per rank: LoRA trains $r(d+k)$ for the adapted layer; full fine-tuning trains every weight in the net. — _Each unit of rank adds one $d$-vector ($B$) and one $k$-vector ($A$), i.e. $d+k$ parameters, so the count grows linearly in $r$._
- Adapt at each $r$ and read held-out accuracy. Expect very low $r$ to underfit, then accuracy to plateau once $r$ is large enough to express the task's update. — _If the needed change has low intrinsic rank, a small $r$ already captures it; adding more rank past that point buys little (&sect;7.2)._
- Compare the plateau to full fine-tuning's accuracy at its (much larger) parameter count. — _The headline of LoRA is matching full fine-tuning with a tiny fraction of trainable parameters._

**Answer:** Trainable parameters grow linearly with $r$ (each rank adds $d+k$). Accuracy rises with $r$
                 and then plateaus: in our run rank $1$ (64 trainable params) reaches only ~0.64 test accuracy,
                 but rank $2$ (128 params) already hits ~0.99 &mdash; matching full fine-tuning's ~1.0 while
                 training about 11&times; fewer parameters than full fine-tuning's 1476. Past rank $2$,
                 more rank adds parameters without improving accuracy. The lesson: pick the smallest $r$ that
                 closes the gap; the task's needed update is low-rank.

</details>

**Problem 2.** Why does LoRA initialize $B$ to zero and $A$ to random, rather than both random or both zero?
            Reason through what the update and its gradient are at step 0 in each case.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Both random: $B A \neq 0$ at init, so the adapted model $W_0 x + \tfrac{\alpha}{r}BAx$ differs from the pretrained model from the start. — _You begin from a random perturbation of the good pretrained weights, which can hurt before training recovers._
- Both zero: $B A = 0$ and the gradient of the loss w.r.t. $B$ is proportional to $A x = 0$, so $B$ cannot move; symmetrically $A$ cannot move. The update is frozen at zero. — _Gradient through the bottleneck needs at least one factor nonzero to be nonzero._
- $B = 0$, $A$ random: $B A = 0$ at init (model = pretrained), but $\partial \text{loss}/\partial B \propto A x \neq 0$, so $B$ moves on step 1 and the update grows from zero. — _Best of both: start from the known-good point AND keep gradients flowing._

**Answer:** $B = 0$, $A$ random is the only choice that both (a) makes $\Delta W = B A = 0$ at init &mdash;
                 so the adapted model equals the pretrained model and training starts from a good point &mdash;
                 and (b) keeps a nonzero gradient on $B$ (it depends on $A x$, and $A$ is random) so the factors
                 can actually learn. Both-random loses (a); both-zero loses (b) and the update is stuck at zero
                 forever. The paper (&sect;4.1) states it directly: random Gaussian $A$, zero $B$, so
                 $\Delta W = B A$ is zero at the beginning of training.

</details>

**Problem 3.** In the worked example you had $d = 4$, $k = 3$, rank $r = 1$, with
            $B = [1, 0, 2, -1]^\top$ and $A = [1, 0, -1]$, giving $B A$ rank-$1$ and $7$ trainable params versus
            $12$ for full fine-tuning. Now suppose the layer is realistic: $d = k = 1000$. Compute the trainable
            parameter counts for full fine-tuning and for LoRA at $r = 1$ and $r = 8$. What is the ratio?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Full fine-tuning trains all of $W_0$: $d \times k = 1000 \times 1000 = 1{,}000{,}000$ parameters. — _Every entry of the $d\times k$ weight matrix is free._
- LoRA at $r$ trains $B$ ($d\times r$) plus $A$ ($r\times k$): $r(d+k) = r(1000+1000) = 2000\,r$. So $r=1$ gives $2000$; $r=8$ gives $16{,}000$. — _Each rank adds one column to $B$ and one row to $A$, i.e. $d+k = 2000$ parameters._
- Ratio to full: $r=1$ trains $2000/1{,}000{,}000 = 0.2\%$; $r=8$ trains $16{,}000/1{,}000{,}000 = 1.6\%$. — _The saving grows with the layer size: $r(d+k)$ shrinks relative to $d\,k$ as $d,k$ grow._

**Answer:** Full fine-tuning trains $1{,}000{,}000$ parameters. LoRA trains $r(d+k) = 2000\,r$:
                 $2{,}000$ at $r=1$ ($0.2\%$ of full) and $16{,}000$ at $r=8$ ($1.6\%$ of full). The bigger the
                 layer, the more lopsided the saving &mdash; which is exactly why LoRA's reductions explode at
                 GPT-3 scale (the abstract quotes 10,000&times; fewer trainable parameters). The toy
                 $4\times 3$ example only saved $7$ vs $12$ because $d, k$ were tiny; the mechanism is the same,
                 the payoff scales with $d\,k$.

</details>